# 4G LTE Throughput Prediction and Signal Quality Analysis

## Project Objective

This project investigates how radio-channel indicators and user-context
variables relate to downlink throughput in a 4G-focused mobile measurement
dataset. The main machine learning objective is to predict downlink throughput
using available radio and contextual features.

## Dataset Provenance

The dataset is based on the research dataset:

[**Beyond Throughput: a 4G LTE Dataset with Channel and Context Metrics**](https://doi.org/10.1145/3204949.3208123)

Authors: Darijo Raca, Jason J. Quinlan, Ahmed H. Zahran, and Cormac J. Sreenan  
Conference: ACM Multimedia Systems Conference, 2018  
DOI: `10.1145/3204949.3208123`

The dataset contains client-side mobile-network measurements collected from
two Irish operators under several mobility conditions. Although the dataset is
4G-focused, some traces also contain 2G and 3G observations. The exact modeling
scope will be decided after the data audit.

## Initial Scope

- Task type: Supervised machine learning
- Problem type: Regression
- Candidate target: `DL_bitrate` (kbit/s)
- Unit of observation: One timestamped network measurement
- Grouping unit: Measurement trace or session

## Important Limitation

The measurements were collected under specific operators, locations, devices,
and mobility scenarios. Therefore, conclusions from this project should not be
generalized automatically to all LTE networks.

For the telecom concepts behind the columns, see the companion
[domain notes](./README.md).

In [1]:
import sys

print(f"Python version: {sys.version.split()[0]}")

Python version: 3.12.13


In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

In [3]:
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
else:
    PROJECT_ROOT = CURRENT_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "Dataset"

if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(f"Raw data directory not found: {RAW_DATA_DIR}")

print("Project directories are configured successfully.")

Project directories are configured successfully.


In [4]:
csv_files = sorted(RAW_DATA_DIR.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError("No CSV files found in data directory")

print(f"Found {len(csv_files)} CSV files.")

Found 135 CSV files.


In [5]:
file_inventory = pd.DataFrame(
    {
        "File Name": [f.name for f in csv_files],
        "Relative Path": [f.relative_to(PROJECT_ROOT).as_posix() for f in csv_files],
        "Size (KiB)": [round(f.stat().st_size / 1024, 2) for f in csv_files],
    }
)

file_inventory_preview = file_inventory.head(10).copy()
file_inventory_preview.index = range(1, len(file_inventory_preview) + 1)

display(
    file_inventory_preview.style.format({"Size (KiB)": "{:,.2f}"})
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
)

,File Name,Relative Path,Size (KiB)
1,A_2017.11.30_16.48.26.csv,data/Dataset/bus/A_2017.11.30_16.48.26.csv,113.78
2,A_2018.01.25_16.33.53.csv,data/Dataset/bus/A_2018.01.25_16.33.53.csv,64.82
3,A_2018.01.25_17.27.30.csv,data/Dataset/bus/A_2018.01.25_17.27.30.csv,108.77
4,A_2018.01.25_18.02.07.csv,data/Dataset/bus/A_2018.01.25_18.02.07.csv,44.85
5,A_2018.01.25_19.50.40.csv,data/Dataset/bus/A_2018.01.25_19.50.40.csv,42.66
6,A_2018.01.26_11.26.26.csv,data/Dataset/bus/A_2018.01.26_11.26.26.csv,179.19
7,A_2018.01.27_10.58.49.csv,data/Dataset/bus/A_2018.01.27_10.58.49.csv,56.26
8,A_2018.01.27_11.12.23.csv,data/Dataset/bus/A_2018.01.27_11.12.23.csv,66.41
9,A_2018.01.27_12.10.00.csv,data/Dataset/bus/A_2018.01.27_12.10.00.csv,64.46
10,B_2018.01.25_16.33.45.csv,data/Dataset/bus/B_2018.01.25_16.33.45.csv,73.61


## Inspect One Trace

Before combining all 135 traces, I want to open one file and understand its
shape, columns, and observations. This keeps the first step concrete and makes
it easier to notice trace-level details that a merged table could hide.

In [6]:
sample_file = csv_files[0]

sample_df = pd.read_csv(sample_file)

print(f"Sample file: {sample_file.name}")
print(f"Trace shape: {sample_df.shape[0]:,} rows × {sample_df.shape[1]} columns")

Sample file: A_2017.11.30_16.48.26.csv
Trace shape: 910 rows × 20 columns


In [7]:
sample_path = sample_file.relative_to(PROJECT_ROOT).as_posix()
display(Markdown(f"**Sample trace:** `{sample_path}`"))

display(Markdown("### First 5 Rows"))
display(sample_df.head(5))

display(Markdown("### Last 5 Rows"))
display(sample_df.tail(5))

**Sample trace:** `data/Dataset/bus/A_2017.11.30_16.48.26.csv`

### First 5 Rows

,Timestamp,Longitude,Latitude,Speed,Operatorname,CellID,NetworkMode,RSRP,RSRQ,SNR,CQI,RSSI,DL_bitrate,UL_bitrate,State,NRxRSRP,NRxRSRQ,ServingCell_Lon,ServingCell_Lat,ServingCell_Distance
0,2017.11.30_16.48.26,-8.501373,51.893359,0,A,2,LTE,-102,-12,10.0,7,-85,3,7,D,-,-,-8.491719,51.893905,665.24000000000001
1,2017.11.30_16.48.26,-8.501291,51.893462,1,A,2,LTE,-102,-12,10.0,7,-85,3,7,D,-,-,-8.491719,51.893905,658.67999999999995
2,2017.11.30_16.48.27,-8.501291,51.893462,1,A,2,LTE,-102,-12,7.0,10,-87,310,14,D,-,-,-8.491719,51.893905,658.67999999999995
3,2017.11.30_16.48.28,-8.501291,51.893462,1,A,2,LTE,-102,-12,7.0,7,-85,0,0,I,-,-,-8.491719,51.893905,658.67999999999995
4,2017.11.30_16.48.29,-8.501291,51.893462,1,A,2,LTE,-102,-13,8.0,7,-85,0,0,I,-,-,-8.491719,51.893905,658.67999999999995


### Last 5 Rows

,Timestamp,Longitude,Latitude,Speed,Operatorname,CellID,NetworkMode,RSRP,RSRQ,SNR,CQI,RSSI,DL_bitrate,UL_bitrate,State,NRxRSRP,NRxRSRQ,ServingCell_Lon,ServingCell_Lat,ServingCell_Distance
905,2017.11.30_17.04.24,-8.556712,51.892251,46,A,1,LTE,-91,-12,3.0,12,-72,7068,128,D,-89.0,-10.0,-8.535593,51.880268,1968.8299999999999
906,2017.11.30_17.04.25,-8.556712,51.892251,46,A,1,LTE,-91,-12,3.0,10,-69,8992,166,D,-89.0,-10.0,-8.535593,51.880268,1968.8299999999999
907,2017.11.30_17.04.26,-8.556712,51.892251,46,A,1,LTE,-86,-11,5.0,9,-76,9584,177,D,-85.0,-9.0,-8.535593,51.880268,1968.8299999999999
908,2017.11.30_17.04.27,-8.556712,51.892251,46,A,1,LTE,-86,-11,5.0,10,-74,11060,198,D,-85.0,-9.0,-8.535593,51.880268,1968.8299999999999
909,2017.11.30_17.04.27,-8.556712,51.892251,46,A,1,LTE,-86,-11,5.0,5,-76,11060,198,D,-85.0,-9.0,-8.535593,51.880268,1968.8299999999999


In [8]:
category_counts = (
    pd.Series(
        [file_path.parent.name for file_path in csv_files],
        name="Category",
    )
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="CSV Files")
    .sort_values("Category")
    .reset_index(drop=True)
)

category_counts_preview = category_counts.copy()
category_counts_preview["Category"] = (
    category_counts_preview["Category"].str.replace("_", " ").str.title()
)

total_row = pd.DataFrame(
    {
        "Category": ["Total"],
        "CSV Files": [category_counts_preview["CSV Files"].sum()],
    }
)

category_counts_preview = pd.concat(
    [category_counts_preview, total_row],
    ignore_index=True,
)

category_counts_preview.index = range(1, len(category_counts_preview) + 1)
category_counts_preview.index.name = "No."

display(Markdown("### CSV Files by Category"))

display(
    category_counts_preview.style.format({"CSV Files": "{:,.0f}"})
    .set_properties(subset=["Category"], **{"text-align": "left"})
    .set_properties(subset=["CSV Files"], **{"text-align": "right"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
)

### CSV Files by Category

,Category,CSV Files
No.,,
1,Bus,16
2,Car,53
3,Pedestrian,31
4,Static,15
5,Train,20
6,Total,135


In [9]:
schema_rows = []

for file_path in csv_files:
    columns = tuple(pd.read_csv(file_path, nrows=0).columns)

    schema_rows.append(
        {
            "Category": file_path.parent.name,
            "File Name": file_path.name,
            "Column Count": len(columns),
            "Columns": columns,
        }
    )

schema_check = pd.DataFrame(schema_rows)

schema_summary = schema_check.groupby("Category", as_index=False).agg(
    Total_Files=("File Name", "count"),
    Column_Count=("Column Count", "first"),
    Unique_Column_Counts=("Column Count", "nunique"),
    Unique_Schemas=("Columns", "nunique"),
)

schema_summary["Same Column Count"] = schema_summary["Unique_Column_Counts"] == 1

schema_summary["Same Column Names and Order"] = schema_summary["Unique_Schemas"] == 1

schema_summary_preview = schema_summary[
    [
        "Category",
        "Total_Files",
        "Column_Count",
        "Same Column Count",
        "Same Column Names and Order",
    ]
].copy()

schema_summary_preview.index = range(1, len(schema_summary_preview) + 1)

display(schema_summary_preview)

,Category,Total_Files,Column_Count,Same Column Count,Same Column Names and Order
1,bus,16,20,True,True
2,car,53,20,True,True
3,pedestrian,31,20,True,True
4,static,15,20,True,True
5,train,20,20,True,True


In [10]:
combined_parts = []

for trace_number, file_path in enumerate(csv_files, start=1):
    trace_df = pd.read_csv(file_path)

    trace_df.insert(0, "Trace ID", f"trace_{trace_number:03d}")
    trace_df.insert(1, "Source Category", file_path.parent.name)
    trace_df.insert(2, "Source File", file_path.name)

    combined_parts.append(trace_df)

combined_df = pd.concat(combined_parts, ignore_index=True)

print(f"Combined shape: {combined_df.shape[0]:,} rows × {combined_df.shape[1]} columns")

Combined shape: 174,523 rows × 23 columns


In [11]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

combined_file = PROCESSED_DIR / "4g_lte_throughput_combined.csv"
combined_df.to_csv(combined_file, index=False)

print(f"Saved to: {combined_file.relative_to(PROJECT_ROOT)}")

Saved to: data\processed\4g_lte_throughput_combined.csv


In [12]:
sample_trace_ids = combined_df["Trace ID"].drop_duplicates().head(3)

preview_columns = [
    "Trace ID",
    "Source Category",
    "Source File",
    *combined_df.columns[3:8],
]

combined_preview = (
    combined_df.loc[
        combined_df["Trace ID"].isin(sample_trace_ids),
        preview_columns,
    ]
    .groupby("Trace ID", group_keys=False)
    .head(3)
    .copy()
)

combined_preview.index = range(1, len(combined_preview) + 1)

display(combined_preview)

,Trace ID,Source Category,Source File,Timestamp,Longitude,Latitude,Speed,Operatorname
1,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.48.26,-8.501373,51.893359,0,A
2,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.48.26,-8.501291,51.893462,1,A
3,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.48.27,-8.501291,51.893462,1,A
4,trace_002,bus,A_2018.01.25_16.33.53.csv,2018.01.25_16.33.53,-8.499627,51.893860,0,A
5,trace_002,bus,A_2018.01.25_16.33.53.csv,2018.01.25_16.33.54,-8.498502,51.893994,34,A
6,trace_002,bus,A_2018.01.25_16.33.53.csv,2018.01.25_16.33.54,-8.498502,51.893994,34,A
7,trace_003,bus,A_2018.01.25_17.27.30.csv,2018.01.25_17.27.30,-8.473233,51.898205,2,A
8,trace_003,bus,A_2018.01.25_17.27.30.csv,2018.01.25_17.27.30,-8.473233,51.898205,2,A
9,trace_003,bus,A_2018.01.25_17.27.30.csv,2018.01.25_17.27.31,-8.473518,51.898446,0,A


In [13]:
total_rows = combined_df.shape[0]
total_columns = combined_df.shape[1]

print(f"Total rows   : {total_rows:,}")
print(f"Total columns: {total_columns}")

Total rows   : 174,523
Total columns: 23


In [14]:
column_types = pd.DataFrame(
    {
        "Column": combined_df.columns,
        "Data Type": combined_df.dtypes.astype(str).values,
    }
)

column_types_preview = column_types.copy()
column_types_preview.index = range(1, len(column_types_preview) + 1)

display(column_types_preview)

,Column,Data Type
1,Trace ID,str
2,Source Category,str
3,Source File,str
4,Timestamp,str
5,Longitude,float64
6,Latitude,float64
7,Speed,int64
8,Operatorname,str
9,CellID,int64
10,NetworkMode,str


In [15]:
combined_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 174523 entries, 0 to 174522
Data columns (total 23 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Trace ID              174523 non-null  str    
 1   Source Category       174523 non-null  str    
 2   Source File           174523 non-null  str    
 3   Timestamp             174523 non-null  str    
 4   Longitude             174523 non-null  float64
 5   Latitude              174523 non-null  float64
 6   Speed                 174523 non-null  int64  
 7   Operatorname          174523 non-null  str    
 8   CellID                174523 non-null  int64  
 9   NetworkMode           174523 non-null  str    
 10  RSRP                  174523 non-null  int64  
 11  RSRQ                  174523 non-null  object 
 12  SNR                   174523 non-null  object 
 13  CQI                   174523 non-null  object 
 14  RSSI                  174523 non-null  object 
 15  DL_bitrate 

In [16]:
numeric_summary = combined_df.describe(include="number")
numeric_summary = numeric_summary.T
numeric_summary = numeric_summary.round(2)

numeric_summary_preview = numeric_summary.copy()
numeric_summary_preview.index.name = "Column"

display(numeric_summary_preview)

,count,mean,std,min,25%,50%,75%,max
Column,,,,,,,,
Longitude,174523.0,-8.44,0.42,-9.59,-8.51,-8.49,-8.46,-6.30
Latitude,174523.0,52.02,0.31,51.87,51.89,51.90,51.94,53.35
Speed,174523.0,30.53,38.30,0.00,0.00,17.00,46.00,166.00
CellID,174523.0,4563.57,12035.05,0.00,2.00,2.00,8.00,65353.00
RSRP,174523.0,-91.52,16.34,-200.00,-102.00,-93.00,-82.00,-27.00
DL_bitrate,174523.0,10822.32,14149.65,0.00,1386.00,5370.00,14796.00,173016.00
UL_bitrate,174523.0,184.75,220.54,0.00,32.00,107.00,266.00,4178.00


In [17]:
string_columns = []

for column in combined_df.columns:
    data_type = str(combined_df[column].dtype)

    if data_type in ["str", "string"]:
        string_columns.append(column)

for column in string_columns:
    value_counts = combined_df[column].value_counts(dropna=False)
    value_counts = value_counts.reset_index()
    value_counts.columns = ["Value", "Count"]

    value_counts_preview = value_counts.copy()
    value_counts_preview.index = range(1, len(value_counts_preview) + 1)

    display(Markdown(f"### {column}"))
    display(value_counts_preview)

### Trace ID

,Value,Count
1,trace_117,12393
2,trace_116,9451
3,trace_056,3007
4,trace_068,2931
5,trace_093,2868
...,...,...
131,trace_012,424
132,trace_004,414
133,trace_039,397
134,trace_122,397


### Source Category

,Value,Count
1,car,75874
2,train,38976
3,pedestrian,33629
4,static,15261
5,bus,10783


### Source File

...

,Value,Count
1,A,139813
2,B,34430
3,0,169
4,27202,55
5,27205,37
6,27201,11
7,27203,8


### NetworkMode

,Value,Count
1,LTE,120824
2,HSPA+,42806
3,HSUPA,8039
4,UMTS,1607
5,EDGE,1240
6,HSDPA,5
7,GPRS,2


### State

,Value,Count
1,D,160336
2,I,14187


In [18]:
object_columns = combined_df.columns[combined_df.dtypes == "object"]

summary_rows = []

for column in object_columns:
    values = combined_df[column]
    text = values.astype("string").str.strip()

    numeric = pd.to_numeric(
        text,
        errors="coerce",
    ).notna()

    placeholder = text.isin(["-", "--", ""])
    missing = values.isna()
    other = ~(numeric | placeholder | missing)

    placeholder_examples = (
        text[placeholder].replace("", "(empty string)").drop_duplicates().tolist()
    )

    total = numeric.sum() + placeholder.sum() + missing.sum() + other.sum()

    summary_rows.append(
        {
            "Column": column,
            "Numeric Values": numeric.sum(),
            "Placeholder Values": placeholder.sum(),
            "Observed Placeholders": ", ".join(placeholder_examples) or "None",
            "Missing Values": missing.sum(),
            "Other Values": other.sum(),
            "Total Values": total,
            "Matches Total Rows": total == len(combined_df),
        }
    )

object_summary = pd.DataFrame(summary_rows)

object_summary_preview = object_summary.copy()
object_summary_preview.index = range(
    1,
    len(object_summary_preview) + 1,
)
object_summary_preview.index.name = "No."

display(object_summary_preview)

,Column,Numeric Values,Placeholder Values,Observed Placeholders,Missing Values,Other Values,Total Values,Matches Total Rows
No.,,,,,,,,
1,RSRQ,173287,1236,-,0,0,174523,True
2,SNR,120801,53722,-,0,0,174523,True
3,CQI,120804,53719,-,0,0,174523,True
4,RSSI,109460,65063,-,0,0,174523,True
5,NRxRSRP,119866,54657,-,0,0,174523,True
6,NRxRSRQ,118649,55874,-,0,0,174523,True
7,ServingCell_Lon,123941,50582,-,0,0,174523,True
8,ServingCell_Lat,123941,50582,-,0,0,174523,True
9,ServingCell_Distance,123941,50582,-,0,0,174523,True


In [19]:
# Exact duplicate rows
duplicate_rows = combined_df[combined_df.duplicated(keep=False)].copy()

print(f"Rows involved in exact duplicates: {len(duplicate_rows):,}")

if not duplicate_rows.empty:
    duplicate_rows_preview = duplicate_rows.head(10).copy()
    duplicate_rows_preview.index = range(
        1,
        len(duplicate_rows_preview) + 1,
    )

    display(duplicate_rows_preview)


# Repeated timestamps within the same trace
timestamp_columns = [
    "Trace ID",
    "Source Category",
    "Source File",
    "Timestamp",
]

repeated_timestamps = combined_df[
    combined_df.duplicated(
        subset=["Trace ID", "Timestamp"],
        keep=False,
    )
][timestamp_columns].copy()

print(f"Rows with repeated timestamps: {len(repeated_timestamps):,}")

if not repeated_timestamps.empty:
    repeated_timestamps_preview = repeated_timestamps.head(10).copy()
    repeated_timestamps_preview.index = range(
        1,
        len(repeated_timestamps_preview) + 1,
    )

    display(repeated_timestamps_preview)

Rows involved in exact duplicates: 1,662


,Trace ID,Source Category,Source File,Timestamp,Longitude,Latitude,Speed,Operatorname,CellID,NetworkMode,...,CQI,RSSI,DL_bitrate,UL_bitrate,State,NRxRSRP,NRxRSRQ,ServingCell_Lon,ServingCell_Lat,ServingCell_Distance
1,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.54.56,-8.511436,51.888744,20,A,2,LTE,...,8,-84,8298,197,D,-101.0,-9.0,-8.508026,51.885559,424.5
2,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.54.56,-8.511436,51.888744,20,A,2,LTE,...,8,-84,8298,197,D,-101.0,-9.0,-8.508026,51.885559,424.5
3,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_17.02.04,-8.535286,51.889096,36,A,2,LTE,...,15,-60,22166,513,D,-,-,-8.435135,51.846654,8340.0799999999999
4,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_17.02.04,-8.535286,51.889096,36,A,2,LTE,...,15,-60,22166,513,D,-,-,-8.435135,51.846654,8340.0799999999999
5,trace_002,bus,A_2018.01.25_16.33.53.csv,2018.01.25_16.39.15,-8.483574,51.897525,17,A,2,HSPA+,...,-,-,917,52,D,-51.0,-24.0,-8.476424,51.90291,774.08000000000004
6,trace_002,bus,A_2018.01.25_16.33.53.csv,2018.01.25_16.39.15,-8.483574,51.897525,17,A,2,HSPA+,...,-,-,917,52,D,-51.0,-24.0,-8.476424,51.90291,774.08000000000004
7,trace_002,bus,A_2018.01.25_16.33.53.csv,2018.01.25_16.44.33,-8.483574,51.897525,17,A,2,HSPA+,...,-,-,11208,180,D,-51.0,-24.0,-8.478049,51.89685,386.45999999999998
8,trace_002,bus,A_2018.01.25_16.33.53.csv,2018.01.25_16.44.33,-8.483574,51.897525,17,A,2,HSPA+,...,-,-,11208,180,D,-51.0,-24.0,-8.478049,51.89685,386.45999999999998
9,trace_003,bus,A_2018.01.25_17.27.30.csv,2018.01.25_17.44.14,-8.489030,51.893012,0,0,8,LTE,...,10,-,0,0,I,-,-,-,-,-
10,trace_003,bus,A_2018.01.25_17.27.30.csv,2018.01.25_17.44.14,-8.489030,51.893012,0,0,8,LTE,...,10,-,0,0,I,-,-,-,-,-


Rows with repeated timestamps: 10,587


,Trace ID,Source Category,Source File,Timestamp
1,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.48.26
2,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.48.26
3,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.49.13
4,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.49.13
5,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.49.44
6,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.49.44
7,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.53.04
8,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.53.04
9,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.53.18
10,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.53.18


## Overview Findings

All 135 CSV traces share the same 20-column schema. After adding `Trace ID`,
`Source Category`, and `Source File`, the combined dataset contains **174,523
rows and 23 columns**. The trace distribution is 16 bus, 53 car, 31 pedestrian,
15 static, and 20 train traces.

### Unique Values

| Column | Unique Values |
| --- | ---: |
| `Trace ID` | 135 |
| `Source Category` | 5 |
| `Source File` | 134 |
| `Timestamp` | 142,130 |
| `Longitude` | 5,647 |
| `Latitude` | 5,372 |
| `Speed` | 167 |
| `Operatorname` | 7 |
| `CellID` | 599 |
| `NetworkMode` | 7 |
| `RSRP` | 90 |
| `RSRQ` | 55 |
| `SNR` | 148 |
| `CQI` | 31 |
| `RSSI` | 82 |
| `DL_bitrate` | 12,545 |
| `UL_bitrate` | 1,619 |
| `State` | 2 |
| `NRxRSRP` | 175 |
| `NRxRSRQ` | 69 |
| `ServingCell_Lon` | 746 |
| `ServingCell_Lat` | 733 |
| `ServingCell_Distance` | 5,395 |

The unique-value counts include `-` when it appears as an observed placeholder.

### Numeric Ranges

| Column | Minimum | Maximum |
| --- | ---: | ---: |
| `Longitude` | -9.59 | -6.30 |
| `Latitude` | 51.87 | 53.35 |
| `Speed` | 0 km/h | 166 km/h |
| `CellID` | 0 | 65,353 |
| `RSRP` | -200 dBm | -27 dBm |
| `DL_bitrate` | 0 kbit/s | 173,016 kbit/s |
| `UL_bitrate` | 0 kbit/s | 4,178 kbit/s |

### Observed Data Notes

- `Speed` contains no negative values; its observed range is 0–166 km/h.
- `RSRP` spans -200 to -27 dBm. These extremes are recorded as findings without
  assuming that they are automatically normal or erroneous.
- `CellID` is an identifier, so its numeric range does not represent cell quality.
- All columns report 174,523 non-null entries, but nine `object` columns use `-`
  for unavailable measurements. Therefore, non-null does not always mean that a
  numeric measurement is available.
- `Operatorname` contains seven recorded labels: `A`, `B`, `0`, `27201`, `27202`,
  `27203`, and `27205`.
- Seven network modes appear: `LTE`, `HSPA+`, `HSUPA`, `UMTS`, `EDGE`, `HSDPA`,
  and `GPRS`.
- `State` contains `D` for 160,336 rows and `I` for 14,187 rows.
- There are 135 trace IDs but 134 unique source filenames because
  `B_2018.01.17_15.56.48.csv` appears once under `car` and once under `train`.
- Exact duplicates involve 1,662 rows, representing 831 duplicated row patterns.
- Repeated timestamps occur in 10,587 rows within the same trace. They are not
  automatically exact duplicates because other measurement values can differ.

---

## Basic Data Preparation

This section prepares a separate working copy while preserving the original
combined table in `combined_df`. The changes remain intentionally limited to
data types, exact duplicates, placeholders, and timestamps.

In [20]:
original_rows, original_columns = combined_df.shape

first_rows = combined_df.head(5).copy()
first_rows.index = range(1, 6)

last_rows = combined_df.tail(5).copy()
last_rows.index = range(original_rows - 4, original_rows + 1)

print(f"Original data: {original_rows:,} rows × {original_columns} columns")

with pd.option_context("display.max_columns", None):
    print("\nFirst 5 rows")
    display(first_rows)

    print("Last 5 rows")
    display(last_rows)

Original data: 174,523 rows × 23 columns

First 5 rows


,Trace ID,Source Category,Source File,Timestamp,Longitude,Latitude,Speed,Operatorname,CellID,NetworkMode,RSRP,RSRQ,SNR,CQI,RSSI,DL_bitrate,UL_bitrate,State,NRxRSRP,NRxRSRQ,ServingCell_Lon,ServingCell_Lat,ServingCell_Distance
1,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.48.26,-8.501373,51.893359,0,A,2,LTE,-102,-12,10.0,7,-85,3,7,D,-,-,-8.491719,51.893905,665.24000000000001
2,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.48.26,-8.501291,51.893462,1,A,2,LTE,-102,-12,10.0,7,-85,3,7,D,-,-,-8.491719,51.893905,658.67999999999995
3,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.48.27,-8.501291,51.893462,1,A,2,LTE,-102,-12,7.0,10,-87,310,14,D,-,-,-8.491719,51.893905,658.67999999999995
4,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.48.28,-8.501291,51.893462,1,A,2,LTE,-102,-12,7.0,7,-85,0,0,I,-,-,-8.491719,51.893905,658.67999999999995
5,trace_001,bus,A_2017.11.30_16.48.26.csv,2017.11.30_16.48.29,-8.501291,51.893462,1,A,2,LTE,-102,-13,8.0,7,-85,0,0,I,-,-,-8.491719,51.893905,658.67999999999995


Last 5 rows


,Trace ID,Source Category,Source File,Timestamp,Longitude,Latitude,Speed,Operatorname,CellID,NetworkMode,RSRP,RSRQ,SNR,CQI,RSSI,DL_bitrate,UL_bitrate,State,NRxRSRP,NRxRSRQ,ServingCell_Lon,ServingCell_Lat,ServingCell_Distance
174519,trace_135,train,B_2018.02.05_15.07.31.csv,2018.02.05_15.24.28,-8.490956,51.934883,74,B,1,LTE,-102,-12,8.0,11,-85,35453,646,D,-109.0,-18.0,-8.489758,51.928426,722.66999999999996
174520,trace_135,train,B_2018.02.05_15.07.31.csv,2018.02.05_15.24.29,-8.490956,51.934883,74,B,1,LTE,-101,-12,9.0,11,-85,37746,684,D,-,-,-8.489758,51.928426,722.66999999999996
174521,trace_135,train,B_2018.02.05_15.07.31.csv,2018.02.05_15.24.30,-8.490956,51.934883,74,B,1,LTE,-101,-12,9.0,9,-82,40530,723,D,-,-,-8.489758,51.928426,722.66999999999996
174522,trace_135,train,B_2018.02.05_15.07.31.csv,2018.02.05_15.24.31,-8.490956,51.934883,74,B,1,LTE,-100,-12,12.0,11,-,43103,756,D,-,-,-8.489758,51.928426,722.66999999999996
174523,trace_135,train,B_2018.02.05_15.07.31.csv,2018.02.05_15.24.32,-8.490956,51.934883,74,B,1,LTE,-100,-12,12.0,11,-,43103,756,D,-,-,-8.489758,51.928426,722.66999999999996


In [21]:
numeric_measurement_columns = [
    "RSRQ",
    "SNR",
    "CQI",
    "RSSI",
    "NRxRSRP",
    "NRxRSRQ",
    "ServingCell_Lon",
    "ServingCell_Lat",
    "ServingCell_Distance",
]

cleaned_df = combined_df.copy()

for column in numeric_measurement_columns:
    cleaned_df[column] = pd.to_numeric(
        cleaned_df[column].replace("-", pd.NA),
        errors="raise",
    )

cleaned_df.info()


summary_rows = []

for column in numeric_measurement_columns:
    values = cleaned_df[column]
    text = values.astype("string").str.strip()

    numeric = pd.to_numeric(text, errors="coerce").notna()
    placeholder = text.isin(["-", "--", ""])
    missing = values.isna()
    other = ~(numeric | placeholder | missing)

    observed_placeholders = (
        text[placeholder]
        .replace("", "(empty string)")
        .drop_duplicates()
        .tolist()
    )

    total = numeric.sum() + placeholder.sum() + missing.sum() + other.sum()

    summary_rows.append(
        {
            "Column": column,
            "Numeric Values": numeric.sum(),
            "Placeholder Values": placeholder.sum(),
            "Observed Placeholders": ", ".join(observed_placeholders) or "None",
            "Missing Values": missing.sum(),
            "Other Values": other.sum(),
            "Total Values": total,
            "Matches Total Rows": total == len(cleaned_df),
        }
    )

cleaning_summary = pd.DataFrame(summary_rows)
cleaning_summary.index = range(1, len(cleaning_summary) + 1)
cleaning_summary.index.name = "No."

with pd.option_context("display.max_columns", None):
    display(cleaning_summary)

<class 'pandas.DataFrame'>
RangeIndex: 174523 entries, 0 to 174522
Data columns (total 23 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Trace ID              174523 non-null  str    
 1   Source Category       174523 non-null  str    
 2   Source File           174523 non-null  str    
 3   Timestamp             174523 non-null  str    
 4   Longitude             174523 non-null  float64
 5   Latitude              174523 non-null  float64
 6   Speed                 174523 non-null  int64  
 7   Operatorname          174523 non-null  str    
 8   CellID                174523 non-null  int64  
 9   NetworkMode           174523 non-null  str    
 10  RSRP                  174523 non-null  int64  
 11  RSRQ                  173287 non-null  float64
 12  SNR                   120801 non-null  float64
 13  CQI                   120804 non-null  float64
 14  RSSI                  109460 non-null  float64
 15  DL_bitrate 

,Column,Numeric Values,Placeholder Values,Observed Placeholders,Missing Values,Other Values,Total Values,Matches Total Rows
No.,,,,,,,,
1,RSRQ,173287,0,None,1236,0,174523,True
2,SNR,120801,0,None,53722,0,174523,True
3,CQI,120804,0,None,53719,0,174523,True
4,RSSI,109460,0,None,65063,0,174523,True
5,NRxRSRP,119866,0,None,54657,0,174523,True
6,NRxRSRQ,118649,0,None,55874,0,174523,True
7,ServingCell_Lon,123941,0,None,50582,0,174523,True
8,ServingCell_Lat,123941,0,None,50582,0,174523,True
9,ServingCell_Distance,123941,0,None,50582,0,174523,True


In [22]:
numeric_columns = cleaned_df.select_dtypes(include="number").columns

numeric_summary = pd.DataFrame(
    {
        "Column": numeric_columns,
        "Data Type": cleaned_df[numeric_columns].dtypes.astype(str).values,
        "Available Values": cleaned_df[numeric_columns].count().values,
        "Missing Values": cleaned_df[numeric_columns].isna().sum().values,
        "Minimum": cleaned_df[numeric_columns].min().values,
        "Maximum": cleaned_df[numeric_columns].max().values,
    }
)

numeric_summary = numeric_summary.round(2)
numeric_summary.index = range(1, len(numeric_summary) + 1)
numeric_summary.index.name = "No."

display(numeric_summary)

,Column,Data Type,Available Values,Missing Values,Minimum,Maximum
No.,,,,,,
1,Longitude,float64,174523,0,-9.59,-6.30
2,Latitude,float64,174523,0,51.87,53.35
3,Speed,int64,174523,0,0.00,166.00
4,CellID,int64,174523,0,0.00,65353.00
5,RSRP,int64,174523,0,-200.00,-27.00
6,RSRQ,float64,173287,1236,-24.00,7.00
7,SNR,float64,120801,53722,-30.00,33.00
8,CQI,float64,120804,53719,1.00,15.00
9,RSSI,float64,109460,65063,-94.00,-36.00


### Initial Numeric Range Review

The reference ranges below are guides, not automatic rules for deleting rows. LTE-specific ranges should only be applied to rows where `NetworkMode == "LTE"`.

| Column | Observed Range | Reference or Expected Range | Initial Assessment |
| --- | ---: | ---: | --- |
| `Longitude` | -9.59 to -6.30 | -180 to 180 | Valid coordinates |
| `Latitude` | 51.87 to 53.35 | -90 to 90 | Valid coordinates |
| `Speed` | 0 to 166 km/h | At least 0 | Reasonable overall; review by mobility category |
| `CellID` | 0 to 65,353 | No quality range | Identifier, not a continuous measurement |
| `RSRP` | -200 to -27 dBm | Approximately -140 to -44 dBm for LTE | Review `-200` and `-27` |
| `RSRQ` | -24 to 7 dB | Approximately -19.5 to -3 dB for LTE | Review after filtering LTE rows |
| `SNR` | -30 to 33 dB | Approximately -20 to 30 dB for LTE | Review values outside the reference range |
| `CQI` | 1 to 15 | 0 to 15 | Within the expected range |
| `RSSI` | -94 to -36 dBm | Approximately -113 to -51 dBm in the Android LTE API | Strong values above -51 need review |
| `DL_bitrate` | 0 to 173,016 kbit/s | At least 0; network-dependent | Plausible; interpret zero using `State` |
| `UL_bitrate` | 0 to 4,178 kbit/s | At least 0; network-dependent | Plausible; interpret zero using `State` |
| `NRxRSRP` | -138 to 0 dBm | Similar to LTE RSRP | The value `0` needs attention |
| `NRxRSRQ` | -225 to -2 dB | Similar to LTE RSRQ | Extreme negative values need strong attention |
| `ServingCell_Lon` | -10.36 to -6.13 | -180 to 180 | Valid coordinates |
| `ServingCell_Lat` | 51.47 to 53.61 | -90 to 90 | Valid coordinates |
| `ServingCell_Distance` | 25.42 to 222,739.98 m | At least 0; deployment-dependent | Very large distances need validation |

### Values Requiring the Most Attention

1. `NRxRSRQ`: values such as `-225`, `-129`, and `-112` may be hidden invalid-value indicators.
2. `RSRP`: `-200` appears 230 times, while `-27` appears twice.
3. `NRxRSRP`: the value `0` appears 22 times and is suspicious for an RSRP measurement.
4. `ServingCell_Distance`: 51 rows exceed 100 km, with a maximum of 222.74 km.
5. `RSRQ`, `SNR`, and `RSSI`: evaluate their ranges separately for LTE and non-LTE network modes.
6. `Speed`: 166 km/h is plausible for a train, but the static category also contains unexpectedly high speeds.

The LTE reference ranges are based on [3GPP/ETSI RSRP and RSRQ reporting definitions](https://www.etsi.org/deliver/etsi_ts/136100_136199/136133/11.09.00_60/ts_136133v110900p.pdf) and the [Android LTE signal API](https://developer.android.com/reference/android/telephony/CellSignalStrengthLte). The dataset paper explains that its network modes include 2G, 3G, and 4G, and that serving-cell coordinates come from OpenCellID; therefore, LTE thresholds should not be applied blindly to every row. [Dataset paper](https://cora.ucc.ie/server/api/core/bitstreams/bc4bae7f-d1f2-4a56-8a51-6fd75211bf52/content)

In [23]:
rows_before = len(cleaned_df)
duplicate_rows = cleaned_df.duplicated().sum()

cleaned_df = cleaned_df.drop_duplicates().copy()

rows_after = len(cleaned_df)
duplicates_remaining = cleaned_df.duplicated().sum()

print(f"Rows before              : {rows_before:,}")
print(f"Exact duplicates removed : {duplicate_rows:,}")
print(f"Rows after               : {rows_after:,}")
print(f"Exact duplicates remaining: {duplicates_remaining:,}")

Rows before              : 174,523
Exact duplicates removed : 831
Rows after               : 173,692
Exact duplicates remaining: 0


In [24]:
cleaned_df["Timestamp"] = pd.to_datetime(
    cleaned_df["Timestamp"],
    format="%Y.%m.%d_%H.%M.%S",
    errors="raise",
)

print(f"Data type     : {cleaned_df['Timestamp'].dtype}")
print(f"Missing values: {cleaned_df['Timestamp'].isna().sum():,}")
print(f"Earliest time : {cleaned_df['Timestamp'].min()}")
print(f"Latest time   : {cleaned_df['Timestamp'].max()}")

Data type     : datetime64[us]
Missing values: 0
Earliest time : 2017-11-21 15:03:50
Latest time   : 2018-02-12 16:28:44


In [25]:
before_placeholders = combined_df.eq("-").sum()
after_placeholders = cleaned_df.eq("-").sum()

before_duplicates = combined_df.duplicated().sum()
after_duplicates = cleaned_df.duplicated().sum()


cleaning_actions = pd.DataFrame(
    {
        "Action": [
            "Converted numeric placeholders (-) into missing values",
            "Removed exact duplicates using all columns",
            "Converted Timestamp into datetime",
        ]
    }
)

cleaning_actions.index = range(1, len(cleaning_actions) + 1)
cleaning_actions.index.name = "Step"


before_after = pd.DataFrame(
    {
        "Check": [
            "Total rows",
            "Total columns",
            "Exact duplicates",
            "Placeholder values",
            "Missing values",
            "Timestamp data type",
        ],
        "Before": [
            f"{len(combined_df):,}",
            combined_df.shape[1],
            f"{before_duplicates:,}",
            f"{before_placeholders.sum():,}",
            f"{combined_df.isna().sum().sum():,}",
            str(combined_df["Timestamp"].dtype),
        ],
        "After": [
            f"{len(cleaned_df):,}",
            cleaned_df.shape[1],
            f"{after_duplicates:,}",
            f"{after_placeholders.sum():,}",
            f"{cleaned_df.isna().sum().sum():,}",
            str(cleaned_df["Timestamp"].dtype),
        ],
    }
)

before_after.index = range(1, len(before_after) + 1)
before_after.index.name = "No."


column_summary = pd.DataFrame(
    {
        "Column": cleaned_df.columns,
        "Before Type": combined_df.dtypes.astype(str).values,
        "After Type": cleaned_df.dtypes.astype(str).values,
        "Before Placeholders": before_placeholders.values,
        "After Placeholders": after_placeholders.values,
        "Before Missing": combined_df.isna().sum().values,
        "After Missing": cleaned_df.isna().sum().values,
        "Available Values": cleaned_df.count().values,
        "Unique Values": cleaned_df.nunique(dropna=False).values,
    }
)

column_summary.index = range(1, len(column_summary) + 1)
column_summary.index.name = "No."


print("Completed actions")
display(cleaning_actions)

print("Before and after cleaning")
display(before_after)

print("Final column summary")
with pd.option_context("display.max_columns", None):
    display(column_summary)

Completed actions


,Action
Step,
1,Converted numeric placeholders (-) into missin...
2,Removed exact duplicates using all columns
3,Converted Timestamp into datetime


Before and after cleaning


,Check,Before,After
No.,,,
1,Total rows,"174,523","173,692"
2,Total columns,23,23
3,Exact duplicates,831,0
4,Placeholder values,"436,017",0
5,Missing values,0,"432,423"
6,Timestamp data type,str,datetime64[us]


Final column summary


,Column,Before Type,After Type,Before Placeholders,After Placeholders,Before Missing,After Missing,Available Values,Unique Values
No.,,,,,,,,,
1,Trace ID,str,str,0,0,0,0,173692,135
2,Source Category,str,str,0,0,0,0,173692,5
3,Source File,str,str,0,0,0,0,173692,134
4,Timestamp,str,datetime64[us],0,0,0,0,173692,142130
5,Longitude,float64,float64,0,0,0,0,173692,5647
6,Latitude,float64,float64,0,0,0,0,173692,5372
7,Speed,int64,int64,0,0,0,0,173692,167
8,Operatorname,str,str,0,0,0,0,173692,7
9,CellID,int64,int64,0,0,0,0,173692,599


In [26]:
unique_preview_rows = []

for column in cleaned_df.columns:
    unique_values = cleaned_df[column].drop_duplicates()
    unique_count = len(unique_values)

    preview_limit = 10 if unique_count <= 10 else 5

    preview_values = (
        unique_values.head(preview_limit)
        .astype("string")
        .fillna("Missing")
        .tolist()
    )

    observed_values = ", ".join(preview_values)
    remaining_values = unique_count - len(preview_values)

    if remaining_values > 0:
        observed_values += f" (+{remaining_values:,} more)"

    unique_preview_rows.append(
        {
            "Column": column,
            "Unique Values": unique_count,
            "Observed Values": observed_values,
        }
    )

unique_value_summary = pd.DataFrame(unique_preview_rows)
unique_value_summary.index = range(1, len(unique_value_summary) + 1)
unique_value_summary.index.name = "No."

print("Observed unique values")
with pd.option_context("display.max_colwidth", None):
    display(unique_value_summary)

Observed unique values


,Column,Unique Values,Observed Values
No.,,,
1,Trace ID,135,"trace_001, trace_002, trace_003, trace_004, trace_005 (+130 more)"
2,Source Category,5,"bus, car, pedestrian, static, train"
3,Source File,134,"A_2017.11.30_16.48.26.csv, A_2018.01.25_16.33.53.csv, A_2018.01.25_17.27.30.csv, A_2018.01.25_18.02.07.csv, A_2018.01.25_19.50.40.csv (+129 more)"
4,Timestamp,142130,"2017-11-30 16:48:26, 2017-11-30 16:48:27, 2017-11-30 16:48:28, 2017-11-30 16:48:29, 2017-11-30 16:48:30 (+142,125 more)"
5,Longitude,5647,"-8.501373, -8.501291, -8.502645, -8.504032, -8.505477 (+5,642 more)"
6,Latitude,5372,"51.893359, 51.893462, 51.893069, 51.892728, 51.892538 (+5,367 more)"
7,Speed,167,"0, 1, 16, 14, 13 (+162 more)"
8,Operatorname,7,"A, 27205, 0, B, 27202, 27201, 27203"
9,CellID,599,"2, 6, 1, 8, 0 (+594 more)"


### Basic Cleaning Summary

The prepared dataset contains **173,692 rows and 23 columns**. During this step, I:

- converted `-` placeholders in nine numeric measurement columns to `NaN`;
- converted those columns to numeric data types;
- removed **831 exact duplicate rows** using all 23 columns while keeping non-identical rows that share a timestamp; and
- converted `Timestamp` from text to `datetime`, covering **21 November 2017 to 12 February 2018**.

The prepared data contains no remaining `-` placeholders or exact duplicate rows. After duplicate removal, **432,423 cells contain `NaN`**. These missing values were neither filled nor used to remove rows. At this stage, suspicious measurements—including extreme radio values and unusually large serving-cell distances—were documented but left unchanged until their network mode and context could be examined properly.

`combined_df` preserves the original combined data, while `cleaned_df` contains the current prepared version.

---

## Domain-Aware Data Validation

The remaining checks examine suspicious measurements in their trace, network,
download-state, and mobility context. Values are changed only when the observed
pattern provides a clear reason; uncertain values remain available for later
analysis.

### Serving-Signal Strength and Download State

In [27]:
rsrp_min = -140
rsrp_max = -44

rsrp_review = cleaned_df.sort_values(
    ["Trace ID", "Timestamp"],
    kind="stable",
).copy()

rsrp_review["Previous RSRP"] = (
    rsrp_review.groupby("Trace ID")["RSRP"].shift(1)
)

rsrp_review["Next RSRP"] = (
    rsrp_review.groupby("Trace ID")["RSRP"].shift(-1)
)

rsrp_review = rsrp_review[
    rsrp_review["RSRP"].notna()
    & ~rsrp_review["RSRP"].between(rsrp_min, rsrp_max)
].copy()


rsrp_summary = (
    rsrp_review.groupby("RSRP", as_index=False)
    .agg(
        Rows=("Trace ID", "size"),
        Traces=("Trace ID", "nunique"),
        Earliest=("Timestamp", "min"),
        Latest=("Timestamp", "max"),
    )
)

rsrp_summary.index = range(1, len(rsrp_summary) + 1)
rsrp_summary.index.name = "No."


rsrp_context = (
    rsrp_review.groupby(
        ["RSRP", "NetworkMode", "State", "Source Category"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Rows"})
    .sort_values(["RSRP", "Rows"], ascending=[True, False])
)

rsrp_context.index = range(1, len(rsrp_context) + 1)
rsrp_context.index.name = "No."


preview_columns = [
    "Trace ID",
    "Timestamp",
    "Source Category",
    "NetworkMode",
    "State",
    "Operatorname",
    "CellID",
    "Previous RSRP",
    "RSRP",
    "Next RSRP",
    "RSRQ",
    "SNR",
    "CQI",
    "RSSI",
    "DL_bitrate",
    "Speed",
]

rsrp_preview = (
    rsrp_review.groupby("RSRP", group_keys=False)
    .head(10)[preview_columns]
    .copy()
)

rsrp_preview.index = range(1, len(rsrp_preview) + 1)
rsrp_preview.index.name = "No."


print(f"LTE reference used : {rsrp_min} to {rsrp_max} dBm")
print(f"Rows to review     : {len(rsrp_review):,}")

print("\nSummary by RSRP value")
display(rsrp_summary)

print("Summary by context")
display(rsrp_context)

print("Row preview")
with pd.option_context("display.max_columns", None):
    display(rsrp_preview)

LTE reference used : -140 to -44 dBm
Rows to review     : 229

Summary by RSRP value


,RSRP,Rows,Traces,Earliest,Latest
No.,,,,,
1,-200,227,13,2017-11-24 14:35:40,2018-02-05 13:42:26
2,-27,2,1,2017-11-30 16:15:39,2017-11-30 16:15:40


Summary by context


,RSRP,NetworkMode,State,Source Category,Rows
No.,,,,,
1,-200,LTE,I,bus,160
2,-200,LTE,D,pedestrian,53
3,-200,HSPA+,D,car,5
4,-200,HSPA+,D,train,3
5,-200,LTE,I,pedestrian,3
6,-200,HSUPA,D,train,1
7,-200,LTE,D,car,1
8,-200,UMTS,I,bus,1
9,-27,LTE,D,static,2


Row preview


,Trace ID,Timestamp,Source Category,NetworkMode,State,Operatorname,CellID,Previous RSRP,RSRP,Next RSRP,RSRQ,SNR,CQI,RSSI,DL_bitrate,Speed
No.,,,,,,,,,,,,,,,,
1,trace_003,2018-01-25 17:42:35,bus,UMTS,I,0,9736,-75.0,-200,-97.0,-2.0,NaN,NaN,NaN,0,0
2,trace_003,2018-01-25 17:42:41,bus,LTE,I,0,8,-97.0,-200,-200.0,-16.0,1.0,7.0,NaN,0,0
3,trace_003,2018-01-25 17:42:42,bus,LTE,I,0,8,-200.0,-200,-200.0,-16.0,1.0,7.0,NaN,0,0
4,trace_003,2018-01-25 17:42:43,bus,LTE,I,0,8,-200.0,-200,-200.0,-16.0,1.0,7.0,NaN,0,0
5,trace_003,2018-01-25 17:42:44,bus,LTE,I,0,8,-200.0,-200,-200.0,-16.0,1.0,7.0,NaN,0,0
6,trace_003,2018-01-25 17:42:46,bus,LTE,I,0,8,-200.0,-200,-200.0,-16.0,1.0,7.0,NaN,0,0
7,trace_003,2018-01-25 17:42:47,bus,LTE,I,0,8,-200.0,-200,-200.0,-16.0,1.0,7.0,NaN,0,0
8,trace_003,2018-01-25 17:42:48,bus,LTE,I,0,8,-200.0,-200,-200.0,-16.0,1.0,7.0,NaN,0,0
9,trace_003,2018-01-25 17:42:49,bus,LTE,I,0,8,-200.0,-200,-200.0,-16.0,1.0,7.0,NaN,0,0


In [28]:
category_order = [
    "bus",
    "car",
    "pedestrian",
    "static",
    "train",
]

rsrp_values = sorted(rsrp_review["RSRP"].unique())

rsrp_by_category = pd.crosstab(
    rsrp_review["Source Category"],
    rsrp_review["RSRP"],
)

rsrp_by_category = rsrp_by_category.reindex(
    index=category_order,
    columns=rsrp_values,
    fill_value=0,
)

rsrp_by_category.columns = [
    f"RSRP {int(value)}" for value in rsrp_by_category.columns
]

rsrp_by_category["Total Rows"] = rsrp_by_category.sum(axis=1)

rsrp_by_category["Affected Traces"] = (
    rsrp_review.groupby("Source Category")["Trace ID"]
    .nunique()
    .reindex(category_order, fill_value=0)
)

rsrp_by_category = rsrp_by_category.reset_index()

value_columns = [
    column for column in rsrp_by_category.columns
    if column.startswith("RSRP ")
]

rsrp_by_category = rsrp_by_category[
    [
        "Source Category",
        "Affected Traces",
        *value_columns,
        "Total Rows",
    ]
]

total_row = {
    "Source Category": "Total",
    "Affected Traces": rsrp_review["Trace ID"].nunique(),
    **{
        column: rsrp_by_category[column].sum()
        for column in value_columns
    },
    "Total Rows": len(rsrp_review),
}

rsrp_by_category = pd.concat(
    [rsrp_by_category, pd.DataFrame([total_row])],
    ignore_index=True,
)

rsrp_by_category.index = range(1, len(rsrp_by_category) + 1)
rsrp_by_category.index.name = "No."

display(rsrp_by_category)

,Source Category,Affected Traces,RSRP -200,RSRP -27,Total Rows
No.,,,,,
1,bus,1,161,0,161
2,car,6,6,0,6
3,pedestrian,2,56,0,56
4,static,1,0,2,2
5,train,4,4,0,4
6,Total,14,227,2,229


In [29]:
bus_rsrp_200_rows = cleaned_df[
    cleaned_df["Source Category"].eq("bus")
    & cleaned_df["RSRP"].eq(-200)
].sort_values(
    ["Trace ID", "Timestamp"],
    kind="stable",
)

if bus_rsrp_200_rows.empty:
    raise ValueError("No RSRP -200 value was found in bus data")

target_index = bus_rsrp_200_rows.index[0]
target_trace = bus_rsrp_200_rows.iloc[0]["Trace ID"]

trace_data = cleaned_df[
    cleaned_df["Trace ID"].eq(target_trace)
].sort_values(
    "Timestamp",
    kind="stable",
)

target_position = trace_data.index.get_loc(target_index)

start_position = max(0, target_position - 5)
end_position = min(len(trace_data), target_position + 6)

context_columns = [
    "Trace ID",
    "Timestamp",
    "Source Category",
    "NetworkMode",
    "State",
    "Speed",
    "CellID",
    "RSRP",
    "RSRQ",
    "SNR",
    "CQI",
    "RSSI",
    "DL_bitrate",
]

bus_rsrp_200_context = trace_data.iloc[
    start_position:end_position
][context_columns].copy()

relative_positions = range(
    start_position - target_position,
    end_position - target_position,
)

bus_rsrp_200_context.insert(
    0,
    "Position",
    [
        "Target" if position == 0 else f"{position:+d}"
        for position in relative_positions
    ],
)

bus_rsrp_200_context.index = range(
    1,
    len(bus_rsrp_200_context) + 1,
)
bus_rsrp_200_context.index.name = "No."

print(f"Selected trace : {target_trace}")
print("Context        : 5 rows before and 5 rows after")

with pd.option_context("display.max_columns", None):
    display(bus_rsrp_200_context)

Selected trace : trace_003
Context        : 5 rows before and 5 rows after


,Position,Trace ID,Timestamp,Source Category,NetworkMode,State,Speed,CellID,RSRP,RSRQ,SNR,CQI,RSSI,DL_bitrate
No.,,,,,,,,,,,,,,
1,-5,trace_003,2018-01-25 17:42:29,bus,HSPA+,D,0,9736,-77,-2.0,NaN,NaN,NaN,3355
2,-4,trace_003,2018-01-25 17:42:30,bus,HSPA+,D,0,9736,-71,-2.0,NaN,NaN,NaN,3813
3,-3,trace_003,2018-01-25 17:42:31,bus,HSPA+,D,0,9736,-71,-2.0,NaN,NaN,NaN,313
4,-2,trace_003,2018-01-25 17:42:33,bus,UMTS,I,0,9736,-75,-2.0,NaN,NaN,NaN,0
5,-1,trace_003,2018-01-25 17:42:34,bus,UMTS,I,0,9736,-75,-2.0,NaN,NaN,NaN,0
6,Target,trace_003,2018-01-25 17:42:35,bus,UMTS,I,0,9736,-200,-2.0,NaN,NaN,NaN,0
7,+1,trace_003,2018-01-25 17:42:39,bus,LTE,I,0,8,-97,-16.0,1.0,7.0,NaN,0
8,+2,trace_003,2018-01-25 17:42:41,bus,LTE,I,0,8,-200,-16.0,1.0,7.0,NaN,0
9,+3,trace_003,2018-01-25 17:42:42,bus,LTE,I,0,8,-200,-16.0,1.0,7.0,NaN,0


In [30]:
category_order = [
    "bus",
    "car",
    "pedestrian",
    "static",
    "train",
]

for rsrp_value in [-200, -27]:
    rsrp_data = cleaned_df[
        cleaned_df["RSRP"].eq(rsrp_value)
    ]

    state_by_category = pd.crosstab(
        rsrp_data["Source Category"],
        rsrp_data["State"],
    )

    state_by_category = state_by_category.reindex(
        index=category_order,
        columns=["I", "D"],
        fill_value=0,
    )

    state_by_category["Total"] = state_by_category.sum(axis=1)
    state_by_category.loc["Total"] = state_by_category.sum(axis=0)

    state_by_category = state_by_category.reset_index()
    state_by_category.columns.name = None

    state_by_category.index = range(
        1,
        len(state_by_category) + 1,
    )
    state_by_category.index.name = "No."

    print(f"RSRP {rsrp_value} by category and state")
    display(state_by_category)

RSRP -200 by category and state


,Source Category,I,D,Total
No.,,,,
1,bus,161,0,161
2,car,0,6,6
3,pedestrian,3,53,56
4,static,0,0,0
5,train,0,4,4
6,Total,164,63,227


RSRP -27 by category and state


,Source Category,I,D,Total
No.,,,,
1,bus,0,0,0
2,car,0,0,0
3,pedestrian,0,0,0
4,static,0,2,2
5,train,0,0,0
6,Total,0,2,2


In [31]:
category_order = [
    "bus",
    "car",
    "pedestrian",
    "static",
    "train",
]

all_states_by_category = pd.crosstab(
    cleaned_df["Source Category"],
    cleaned_df["State"],
)

all_states_by_category = all_states_by_category.reindex(
    index=category_order,
    columns=["I", "D"],
    fill_value=0,
)

all_states_by_category["Total"] = (
    all_states_by_category.sum(axis=1)
)

all_states_by_category.loc["Total"] = (
    all_states_by_category.sum(axis=0)
)

all_states_by_category = all_states_by_category.reset_index()
all_states_by_category.columns.name = None

all_states_by_category.index = range(
    1,
    len(all_states_by_category) + 1,
)
all_states_by_category.index.name = "No."

print("All rows by category and state")
display(all_states_by_category)

All rows by category and state


,Source Category,I,D,Total
No.,,,,
1,bus,412,10355,10767
2,car,153,75665,75818
3,pedestrian,177,33329,33506
4,static,104,15145,15249
5,train,12845,25507,38352
6,Total,13691,160001,173692


In [32]:
state_summary_rows = []

for state in ["I", "D"]:
    state_data = cleaned_df[
        cleaned_df["State"].eq(state)
    ]

    state_summary_rows.append(
        {
            "State": state,
            "Total Rows": len(state_data),
            "DL_bitrate = 0": state_data["DL_bitrate"].eq(0).sum(),
            "DL_bitrate > 0": state_data["DL_bitrate"].gt(0).sum(),
        }
    )

state_summary = pd.DataFrame(state_summary_rows)
state_summary.index = range(1, len(state_summary) + 1)
state_summary.index.name = "No."

all_idle_values_are_zero = cleaned_df.loc[
    cleaned_df["State"].eq("I"),
    "DL_bitrate",
].eq(0).all()

downloading_zero_rows = cleaned_df[
    cleaned_df["State"].eq("D")
    & cleaned_df["DL_bitrate"].eq(0)
].shape[0]

display(state_summary)

print(f"All idle rows have zero throughput: {all_idle_values_are_zero}")
print(f"Downloading rows with zero throughput: {downloading_zero_rows:,}")

,State,Total Rows,DL_bitrate = 0,DL_bitrate > 0
No.,,,,
1,I,13691,13691,0
2,D,160001,4087,155914


All idle rows have zero throughput: True


### Download-State Decision

All **13,691 idle rows** have `DL_bitrate = 0`. For the main throughput
analysis, I will preserve these rows in `cleaned_df` but use only observations
with `State = "D"` in the modeling dataset.

The **4,087 downloading rows** with zero throughput will remain because they
represent active-download observations where no downlink throughput was
recorded during that timestamp. After filtering, `State` will not be used as a
model feature because every selected row has the same value.

In [33]:
validated_df = cleaned_df.copy()

invalid_rsrp_values = [-200, -27]

invalid_rsrp_rows = validated_df[
    "RSRP"
].isin(invalid_rsrp_values)

validated_df["RSRP"] = validated_df["RSRP"].astype(float)

validated_df.loc[
    invalid_rsrp_rows,
    "RSRP",
] = pd.NA

print(f"Invalid RSRP values replaced: {invalid_rsrp_rows.sum():,}")
print(f"Missing RSRP values now      : {validated_df['RSRP'].isna().sum():,}")
print(f"Remaining -200 values        : {validated_df['RSRP'].eq(-200).sum():,}")
print(f"Remaining -27 values         : {validated_df['RSRP'].eq(-27).sum():,}")

Invalid RSRP values replaced: 229
Missing RSRP values now      : 229
Remaining -200 values        : 0
Remaining -27 values         : 0


### Serving-Signal Decision

The **227 observations with `RSRP = -200`** and **2 observations with
`RSRP = -27`** were inconsistent with the surrounding measurements and the
expected LTE reporting range. Their rows remain in `validated_df`, but these
**229 RSRP measurements** are represented as `NaN`.

### Neighbour-Cell Measurements

The next checks examine `NRxRSRP` and `NRxRSRQ` as a pair before deciding which
values are reliable enough to retain.

In [34]:
suspicious_nrxrsrp_values = [0]

suspicious_nrxrsrq_values = [
    -225,
    -129,
    -112,
    -110,
    -105,
    -104,
    -77,
]

nrxrsrp_mask = validated_df["NRxRSRP"].isin(
    suspicious_nrxrsrp_values
)

nrxrsrq_mask = validated_df["NRxRSRQ"].isin(
    suspicious_nrxrsrq_values
)

nrx_review = validated_df[
    nrxrsrp_mask | nrxrsrq_mask
].copy()

print(f"Rows with NRxRSRP = 0        : {nrxrsrp_mask.sum():,}")
print(f"Rows with unusual NRxRSRQ    : {nrxrsrq_mask.sum():,}")
print(f"Total unique rows to review  : {len(nrx_review):,}")

Rows with NRxRSRP = 0        : 22
Rows with unusual NRxRSRQ    : 8
Total unique rows to review  : 23


In [35]:
nrxrsrp_summary = (
    nrx_review[
        nrx_review["NRxRSRP"].isin(
            suspicious_nrxrsrp_values
        )
    ]
    .groupby("NRxRSRP", as_index=False)
    .agg(
        Rows=("Trace ID", "size"),
        Traces=("Trace ID", "nunique"),
        Earliest=("Timestamp", "min"),
        Latest=("Timestamp", "max"),
    )
)

nrxrsrq_summary = (
    nrx_review[
        nrx_review["NRxRSRQ"].isin(
            suspicious_nrxrsrq_values
        )
    ]
    .groupby("NRxRSRQ", as_index=False)
    .agg(
        Rows=("Trace ID", "size"),
        Traces=("Trace ID", "nunique"),
        Earliest=("Timestamp", "min"),
        Latest=("Timestamp", "max"),
    )
)

nrxrsrp_summary.index = range(1, len(nrxrsrp_summary) + 1)
nrxrsrq_summary.index = range(1, len(nrxrsrq_summary) + 1)

nrxrsrp_summary.index.name = "No."
nrxrsrq_summary.index.name = "No."

print("Suspicious NRxRSRP values")
display(nrxrsrp_summary)

print("Suspicious NRxRSRQ values")
display(nrxrsrq_summary)

Suspicious NRxRSRP values


,NRxRSRP,Rows,Traces,Earliest,Latest
No.,,,,,
1,0.0,22,6,2017-11-24 14:52:01,2018-02-02 10:28:45


Suspicious NRxRSRQ values


,NRxRSRQ,Rows,Traces,Earliest,Latest
No.,,,,,
1,-225.0,1,1,2018-02-05 15:14:51,2018-02-05 15:14:51
2,-129.0,1,1,2017-12-18 07:58:46,2017-12-18 07:58:46
3,-112.0,2,1,2017-11-25 12:05:26,2017-11-25 13:46:16
4,-110.0,1,1,2017-11-25 12:55:51,2017-11-25 12:55:51
5,-105.0,1,1,2017-11-25 14:19:52,2017-11-25 14:19:52
6,-104.0,1,1,2018-02-02 10:28:45,2018-02-02 10:28:45
7,-77.0,1,1,2017-11-25 12:22:15,2017-11-25 12:22:15


In [36]:
nrx_pair_summary = (
    nrx_review.groupby(
        ["NRxRSRP", "NRxRSRQ"],
        dropna=False,
        as_index=False,
    )
    .agg(
        Rows=("Trace ID", "size"),
        Traces=("Trace ID", "nunique"),
    )
    .sort_values("Rows", ascending=False)
)

nrx_pair_summary.index = range(
    1,
    len(nrx_pair_summary) + 1,
)
nrx_pair_summary.index.name = "No."

display(nrx_pair_summary)

,NRxRSRP,NRxRSRQ,Rows,Traces
No.,,,,
1,0.0,-51.0,15,4
2,0.0,-112.0,2,1
3,0.0,-129.0,1,1
4,-98.0,-225.0,1,1
5,0.0,-110.0,1,1
6,0.0,-105.0,1,1
7,0.0,-104.0,1,1
8,0.0,-77.0,1,1


In [37]:
nrx_context_summary = (
    nrx_review.groupby(
        [
            "Source Category",
            "NetworkMode",
            "State",
        ],
        as_index=False,
    )
    .agg(
        Rows=("Trace ID", "size"),
        Traces=("Trace ID", "nunique"),
    )
    .sort_values("Rows", ascending=False)
)

nrx_context_summary.index = range(
    1,
    len(nrx_context_summary) + 1,
)
nrx_context_summary.index.name = "No."

display(nrx_context_summary)


detail_columns = [
    "Trace ID",
    "Timestamp",
    "Source Category",
    "NetworkMode",
    "State",
    "Operatorname",
    "CellID",
    "RSRP",
    "RSRQ",
    "NRxRSRP",
    "NRxRSRQ",
    "SNR",
    "CQI",
    "RSSI",
    "DL_bitrate",
    "Speed",
]

nrx_detail = (
    nrx_review.sort_values(
        ["Trace ID", "Timestamp"],
        kind="stable",
    )[detail_columns]
    .copy()
)

nrx_detail.index = range(1, len(nrx_detail) + 1)
nrx_detail.index.name = "No."

with pd.option_context(
    "display.max_columns",
    None,
    "display.max_rows",
    None,
):
    display(nrx_detail)

,Source Category,NetworkMode,State,Rows,Traces
No.,,,,,
1,train,HSPA+,D,7,3
2,train,HSPA+,I,6,2
3,train,LTE,I,4,2
4,train,LTE,D,3,3
5,train,HSUPA,I,2,1
6,car,HSPA+,D,1,1


,Trace ID,Timestamp,Source Category,NetworkMode,State,Operatorname,CellID,RSRP,RSRQ,NRxRSRP,NRxRSRQ,SNR,CQI,RSSI,DL_bitrate,Speed
No.,,,,,,,,,,,,,,,,
1,trace_064,2018-01-17 16:13:31,car,HSPA+,D,B,1423,-75.0,-2.0,0.0,-51.0,NaN,NaN,NaN,2538,45
2,trace_116,2017-11-24 14:52:01,train,HSPA+,I,A,12528,-99.0,-2.0,0.0,-51.0,NaN,NaN,NaN,0,125
3,trace_116,2017-11-24 15:09:04,train,HSPA+,D,A,14286,-101.0,-2.0,0.0,-51.0,NaN,NaN,NaN,13,95
4,trace_116,2017-11-24 15:26:14,train,HSPA+,I,A,18612,-85.0,-2.0,0.0,-51.0,NaN,NaN,NaN,0,108
5,trace_116,2017-11-24 15:43:53,train,HSPA+,D,A,18286,-81.0,-2.0,0.0,-51.0,NaN,NaN,NaN,335,142
6,trace_116,2017-11-24 16:01:09,train,HSPA+,I,A,18679,-99.0,-2.0,0.0,-51.0,NaN,NaN,NaN,0,136
7,trace_116,2017-11-24 16:18:24,train,HSPA+,I,A,18221,-99.0,-2.0,0.0,-51.0,NaN,NaN,NaN,0,134
8,trace_116,2017-11-24 16:52:42,train,HSPA+,I,A,15066,-83.0,-2.0,0.0,-51.0,NaN,NaN,NaN,0,89
9,trace_117,2017-11-25 12:05:26,train,LTE,I,A,0,-110.0,-15.0,0.0,-112.0,-4.0,4.0,-94.0,0,0


In [38]:
common_nrxrsrq_values = [-24, -20]

common_nrxrsrq_data = validated_df[
    validated_df["NRxRSRQ"].isin(
        common_nrxrsrq_values
    )
].copy()

common_nrxrsrq_summary = (
    common_nrxrsrq_data.groupby(
        "NRxRSRQ",
        as_index=False,
    )
    .agg(
        Rows=("Trace ID", "size"),
        Traces=("Trace ID", "nunique"),
        Missing_Paired_NRxRSRP=(
            "NRxRSRP",
            lambda values: values.isna().sum(),
        ),
    )
)

common_nrxrsrq_context = (
    common_nrxrsrq_data.groupby(
        [
            "NRxRSRQ",
            "NetworkMode",
            "State",
            "Source Category",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Rows"})
    .sort_values(
        ["NRxRSRQ", "Rows"],
        ascending=[True, False],
    )
)

common_nrxrsrq_summary.index = range(
    1,
    len(common_nrxrsrq_summary) + 1,
)
common_nrxrsrq_context.index = range(
    1,
    len(common_nrxrsrq_context) + 1,
)

common_nrxrsrq_summary.index.name = "No."
common_nrxrsrq_context.index.name = "No."

print("Common NRxRSRQ values")
display(common_nrxrsrq_summary)

print("Common NRxRSRQ values by context")
display(common_nrxrsrq_context)

Common NRxRSRQ values


,NRxRSRQ,Rows,Traces,Missing_Paired_NRxRSRP
No.,,,,
1,-24.0,49621,54,0
2,-20.0,3747,113,0


Common NRxRSRQ values by context


,NRxRSRQ,NetworkMode,State,Source Category,Rows
No.,,,,,
1,-24.0,HSPA+,D,car,17757
2,-24.0,HSPA+,D,train,16513
3,-24.0,HSPA+,I,train,4579
4,-24.0,HSUPA,D,car,4476
5,-24.0,HSUPA,I,train,1394
6,-24.0,HSUPA,D,train,1102
7,-24.0,HSPA+,D,bus,940
8,-24.0,UMTS,D,train,851
9,-24.0,HSUPA,D,pedestrian,718


In [39]:
invalid_nrxrsrp_rows = validated_df["NRxRSRP"].eq(0)

invalid_nrxrsrq_rows = (
    invalid_nrxrsrp_rows
    | validated_df["NRxRSRQ"].eq(-225)
)

missing_before = validated_df[
    ["NRxRSRP", "NRxRSRQ"]
].isna().sum()


validated_df.loc[
    invalid_nrxrsrp_rows,
    "NRxRSRP",
] = pd.NA

validated_df.loc[
    invalid_nrxrsrq_rows,
    "NRxRSRQ",
] = pd.NA


missing_after = validated_df[
    ["NRxRSRP", "NRxRSRQ"]
].isna().sum()

validation_summary = pd.DataFrame(
    {
        "Column": ["NRxRSRP", "NRxRSRQ"],
        "Values Replaced": [
            invalid_nrxrsrp_rows.sum(),
            invalid_nrxrsrq_rows.sum(),
        ],
        "Missing Before": [
            missing_before["NRxRSRP"],
            missing_before["NRxRSRQ"],
        ],
        "Missing After": [
            missing_after["NRxRSRP"],
            missing_after["NRxRSRQ"],
        ],
        "Invalid Values Remaining": [
            validated_df["NRxRSRP"].eq(0).sum(),
            validated_df["NRxRSRQ"].eq(-225).sum(),
        ],
    }
)

validation_summary.index = range(1, len(validation_summary) + 1)
validation_summary.index.name = "No."

display(validation_summary)

,Column,Values Replaced,Missing Before,Missing After,Invalid Values Remaining
No.,,,,,
1,NRxRSRP,22,54502,54524,0
2,NRxRSRQ,23,55655,55678,0


### Neighbour-Cell Decision

`NRxRSRP = 0` occurred in **22 observations** and was treated as an unavailable
neighbour-signal measurement. The paired `NRxRSRQ` values in those rows were
also treated as unavailable. One additional `NRxRSRQ = -225` measurement was
replaced with `NaN`, while its plausible `NRxRSRP = -98` value was retained.

The frequently observed `NRxRSRQ` values of `-20` and `-24` remain unchanged
because their network-mode context does not support treating them as universal
errors.

### Mobility-Speed Validation

The mobility folders describe collection scenarios, but the row-level `Speed`
measurements still require inspection. The static traces provide the clearest
case for checking whether recorded speed behaves like continuous movement or
periodic GPS reporting.

In [40]:
speed_summary = (
    validated_df.groupby(
        "Source Category",
        as_index=False,
    )
    .agg(
        Total_Rows=("Speed", "size"),
        Minimum_Speed=("Speed", "min"),
        Maximum_Speed=("Speed", "max"),
        Missing_Values=("Speed", lambda values: values.isna().sum()),
        Negative_Values=("Speed", lambda values: values.lt(0).sum()),
    )
)

speed_summary.columns = [
    "Source Category",
    "Total Rows",
    "Minimum Speed (km/h)",
    "Maximum Speed (km/h)",
    "Missing Values",
    "Negative Values",
]

speed_summary.index = range(1, len(speed_summary) + 1)
speed_summary.index.name = "No."

display(speed_summary)

,Source Category,Total Rows,Minimum Speed (km/h),Maximum Speed (km/h),Missing Values,Negative Values
No.,,,,,,
1,bus,10767,0,49,0,0
2,car,75818,0,97,0,0
3,pedestrian,33506,0,8,0,0
4,static,15249,0,97,0,0
5,train,38352,0,166,0,0


In [41]:
static_speed_data = validated_df[
    validated_df["Source Category"].eq("static")
].copy()

static_speed_summary = (
    static_speed_data.groupby(
        "Speed",
        as_index=False,
    )
    .agg(
        Rows=("Speed", "size"),
        Traces=("Trace ID", "nunique"),
    )
    .sort_values("Speed")
)

static_speed_summary["Percentage"] = (
    static_speed_summary["Rows"]
    / len(static_speed_data)
    * 100
).round(2)

static_speed_summary.index = range(
    1,
    len(static_speed_summary) + 1,
)
static_speed_summary.index.name = "No."

print(f"Total static rows         : {len(static_speed_data):,}")
print(f"Rows with Speed = 0       : {static_speed_data['Speed'].eq(0).sum():,}")
print(f"Rows with Speed above 0   : {static_speed_data['Speed'].gt(0).sum():,}")

display(static_speed_summary)

Total static rows         : 15,249
Rows with Speed = 0       : 10,031
Rows with Speed above 0   : 5,218


,Speed,Rows,Traces,Percentage
No.,,,,
1,0,10031,15,65.78
2,1,14,1,0.09
3,2,287,1,1.88
4,4,3,1,0.02
5,6,1762,2,11.55
6,8,10,1,0.07
7,9,2168,3,14.22
8,13,15,2,0.10
9,19,3,1,0.02


In [42]:
static_trace_summary = (
    static_speed_data.groupby(
        ["Trace ID", "Source File"],
        as_index=False,
    )
    .agg(
        Total_Rows=("Speed", "size"),
        Minimum_Speed=("Speed", "min"),
        Maximum_Speed=("Speed", "max"),
        Unique_Speeds=("Speed", "nunique"),
    )
    .sort_values(
        "Maximum_Speed",
        ascending=False,
    )
)

static_trace_summary.columns = [
    "Trace ID",
    "Source File",
    "Total Rows",
    "Minimum Speed (km/h)",
    "Maximum Speed (km/h)",
    "Unique Speed Values",
]

static_trace_summary.index = range(
    1,
    len(static_trace_summary) + 1,
)
static_trace_summary.index.name = "No."

display(static_trace_summary)

,Trace ID,Source File,Total Rows,Minimum Speed (km/h),Maximum Speed (km/h),Unique Speed Values
No.,,,,,,
1,trace_102,A_2017.11.23_10.08.29.csv,1000,0,97,6
2,trace_104,A_2017.11.28_11.55.31.csv,1047,0,29,8
3,trace_106,A_2017.11.30_16.15.04.csv,590,0,28,2
4,trace_103,A_2017.11.23_13.14.40.csv,2193,0,9,3
5,trace_108,A_2017.12.03_10.35.01.csv,1093,0,9,2
6,trace_101,A_2017.11.22_10.06.58.csv,2623,0,6,2
7,trace_105,A_2017.11.30_16.15.00.csv,538,0,0,1
8,trace_107,A_2017.12.03_10.09.56.csv,1001,0,0,1
9,trace_109,A_2017.12.15_11.05.30.csv,598,0,0,1


In [43]:
static_speed_97 = static_speed_data[
    static_speed_data["Speed"].eq(97)
][
    [
        "Trace ID",
        "Source File",
        "Timestamp",
        "Longitude",
        "Latitude",
        "Speed",
        "NetworkMode",
        "State",
        "DL_bitrate",
    ]
].copy()

static_speed_97.index = range(
    1,
    len(static_speed_97) + 1,
)
static_speed_97.index.name = "No."

display(static_speed_97)

,Trace ID,Source File,Timestamp,Longitude,Latitude,Speed,NetworkMode,State,DL_bitrate
No.,,,,,,,,,
1,trace_102,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:04,-8.497229,51.893126,97,LTE,D,4216
2,trace_102,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:05,-8.497229,51.893126,97,LTE,D,5556
3,trace_102,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:06,-8.497229,51.893126,97,LTE,D,6386
4,trace_102,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:07,-8.497229,51.893126,97,LTE,D,2236
5,trace_102,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:09,-8.497229,51.893126,97,LTE,D,3769
6,trace_102,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:10,-8.497229,51.893126,97,LTE,D,3109
7,trace_102,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:11,-8.497229,51.893126,97,LTE,D,4564
8,trace_102,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:12,-8.497229,51.893126,97,LTE,D,4574
9,trace_102,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:13,-8.497229,51.893126,97,LTE,D,4596


In [44]:
static_nonzero_speed = validated_df[
    validated_df["Source Category"].eq("static")
    & validated_df["Speed"].gt(0)
].copy()


speed_value_summary = (
    static_nonzero_speed.groupby(
        "Speed",
        as_index=False,
    )
    .agg(
        Rows=("Speed", "size"),
        Traces=("Trace ID", "nunique"),
    )
    .sort_values("Speed")
)

speed_value_summary.index = range(
    1,
    len(speed_value_summary) + 1,
)
speed_value_summary.index.name = "No."


speed_trace_summary = (
    static_nonzero_speed.groupby(
        ["Trace ID", "Source File", "Speed"],
        as_index=False,
    )
    .agg(
        Rows=("Speed", "size"),
        Earliest=("Timestamp", "min"),
        Latest=("Timestamp", "max"),
    )
    .sort_values(
        ["Speed", "Trace ID"],
        ascending=[False, True],
    )
)

speed_trace_summary.index = range(
    1,
    len(speed_trace_summary) + 1,
)
speed_trace_summary.index.name = "No."


print(f"Static rows with Speed above 0: {len(static_nonzero_speed):,}")

print("\nObserved speed values")
display(speed_value_summary)

print("Speed values by trace")
with pd.option_context("display.max_rows", None):
    display(speed_trace_summary)

Static rows with Speed above 0: 5,218

Observed speed values


,Speed,Rows,Traces
No.,,,
1,1,14,1
2,2,287,1
3,4,3,1
4,6,1762,2
5,8,10,1
6,9,2168,3
7,13,15,2
8,19,3,1
9,24,394,1


Speed values by trace


,Trace ID,Source File,Speed,Rows,Earliest,Latest
No.,,,,,,
1,trace_102,A_2017.11.23_10.08.29.csv,97,9,2017-11-23 10:13:04,2017-11-23 10:13:13
2,trace_104,A_2017.11.28_11.55.31.csv,29,2,2017-11-28 11:57:49,2017-11-28 11:57:50
3,trace_106,A_2017.11.30_16.15.04.csv,28,101,2017-11-30 16:24:08,2017-11-30 16:25:59
4,trace_102,A_2017.11.23_10.08.29.csv,27,450,2017-11-23 10:13:13,2017-11-23 10:21:33
5,trace_104,A_2017.11.28_11.55.31.csv,24,394,2017-11-28 12:07:09,2017-11-28 12:14:16
6,trace_102,A_2017.11.23_10.08.29.csv,19,3,2017-11-23 10:26:45,2017-11-23 10:26:47
7,trace_102,A_2017.11.23_10.08.29.csv,13,2,2017-11-23 10:12:50,2017-11-23 10:12:51
8,trace_104,A_2017.11.28_11.55.31.csv,13,13,2017-11-28 11:56:30,2017-11-28 11:56:43
9,trace_103,A_2017.11.23_13.14.40.csv,9,2154,2017-11-23 13:15:06,2017-11-23 13:55:06


In [45]:
selected_trace_id = "trace_102"
selected_speed = 97

target_rows = validated_df[
    validated_df["Trace ID"].eq(selected_trace_id)
    & validated_df["Speed"].eq(selected_speed)
]

start_time = (
    target_rows["Timestamp"].min()
    - pd.Timedelta(seconds=5)
)

end_time = (
    target_rows["Timestamp"].max()
    + pd.Timedelta(seconds=5)
)

speed_97_output = (
    validated_df[
        validated_df["Trace ID"].eq(selected_trace_id)
        & validated_df["Timestamp"].between(
            start_time,
            end_time,
        )
    ]
    .sort_values("Timestamp", kind="stable")
    .copy()
)

speed_97_output.insert(
    0,
    "Row Description",
    [
        "Target: Speed = 97"
        if speed == selected_speed
        else "Surrounding row"
        for speed in speed_97_output["Speed"]
    ],
)

speed_97_output.index = range(
    1,
    len(speed_97_output) + 1,
)
speed_97_output.index.name = "No."

print("Static Speed Review")
print(f"Trace          : {selected_trace_id}")
print(f"Target speed   : {selected_speed} km/h")
print(f"Target rows    : {len(target_rows):,}")
print(f"Displayed from : {start_time}")
print(f"Displayed until: {end_time}")
print("\nAll columns are displayed below.")

with pd.option_context(
    "display.max_columns",
    None,
    "display.max_colwidth",
    None,
):
    display(speed_97_output)

Static Speed Review
Trace          : trace_102
Target speed   : 97 km/h
Target rows    : 9
Displayed from : 2017-11-23 10:12:59
Displayed until: 2017-11-23 10:13:18

All columns are displayed below.


,Row Description,Trace ID,Source Category,Source File,Timestamp,Longitude,Latitude,Speed,Operatorname,CellID,NetworkMode,RSRP,RSRQ,SNR,CQI,RSSI,DL_bitrate,UL_bitrate,State,NRxRSRP,NRxRSRQ,ServingCell_Lon,ServingCell_Lat,ServingCell_Distance
No.,,,,,,,,,,,,,,,,,,,,,,,,
1,Surrounding row,trace_102,static,A_2017.11.23_10.08.29.csv,2017-11-23 10:12:59,-8.498004,51.893446,0,A,8,LTE,-101.0,-15.0,8.0,9.0,-84.0,2527,58,D,-115.0,-12.0,NaN,NaN,NaN
2,Surrounding row,trace_102,static,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:00,-8.498004,51.893446,0,A,8,LTE,-102.0,-15.0,8.0,9.0,-84.0,5032,191,D,-112.0,-13.0,NaN,NaN,NaN
3,Surrounding row,trace_102,static,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:01,-8.498004,51.893446,0,A,8,LTE,-102.0,-15.0,8.0,9.0,-84.0,6945,128,D,-112.0,-13.0,NaN,NaN,NaN
4,Surrounding row,trace_102,static,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:01,-8.499497,51.893164,2,A,8,LTE,-102.0,-15.0,8.0,9.0,-84.0,6945,128,D,-112.0,-13.0,NaN,NaN,NaN
5,Surrounding row,trace_102,static,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:02,-8.499497,51.893164,2,A,8,LTE,-102.0,-16.0,8.0,9.0,-84.0,8322,217,D,-112.0,-13.0,NaN,NaN,NaN
6,Surrounding row,trace_102,static,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:03,-8.499497,51.893164,2,A,8,LTE,-102.0,-16.0,8.0,9.0,-84.0,7552,284,D,-112.0,-13.0,NaN,NaN,NaN
7,Surrounding row,trace_102,static,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:04,-8.499497,51.893164,2,A,8,LTE,-102.0,-15.0,8.0,9.0,-84.0,4216,271,D,-112.0,-13.0,NaN,NaN,NaN
8,Target: Speed = 97,trace_102,static,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:04,-8.497229,51.893126,97,A,8,LTE,-102.0,-15.0,8.0,9.0,-84.0,4216,271,D,-112.0,-13.0,NaN,NaN,NaN
9,Target: Speed = 97,trace_102,static,A_2017.11.23_10.08.29.csv,2017-11-23 10:13:05,-8.497229,51.893126,97,A,8,LTE,-102.0,-15.0,8.0,9.0,-84.0,5556,107,D,-112.0,-13.0,NaN,NaN,NaN


In [46]:
selected_trace_id = "trace_103"
selected_speed = 9

target_rows = validated_df[
    validated_df["Trace ID"].eq(selected_trace_id)
    & validated_df["Speed"].eq(selected_speed)
]

first_time = target_rows["Timestamp"].min()
last_time = target_rows["Timestamp"].max()


speed_9_start = (
    validated_df[
        validated_df["Trace ID"].eq(selected_trace_id)
        & validated_df["Timestamp"].between(
            first_time - pd.Timedelta(seconds=5),
            first_time + pd.Timedelta(seconds=5),
        )
    ]
    .sort_values("Timestamp", kind="stable")
    .copy()
)

speed_9_end = (
    validated_df[
        validated_df["Trace ID"].eq(selected_trace_id)
        & validated_df["Timestamp"].between(
            last_time - pd.Timedelta(seconds=5),
            last_time + pd.Timedelta(seconds=5),
        )
    ]
    .sort_values("Timestamp", kind="stable")
    .copy()
)


for output in [speed_9_start, speed_9_end]:
    output.insert(
        0,
        "Row Description",
        [
            "Target: Speed = 9"
            if speed == selected_speed
            else "Surrounding row"
            for speed in output["Speed"]
        ],
    )

    output.index = range(1, len(output) + 1)
    output.index.name = "No."


print("Static Speed Review")
print(f"Trace             : {selected_trace_id}")
print(f"Target speed      : {selected_speed} km/h")
print(f"Total target rows : {len(target_rows):,}")
print(f"First occurrence  : {first_time}")
print(f"Last occurrence   : {last_time}")

print("\nBeginning of Speed = 9")
with pd.option_context("display.max_columns", None):
    display(speed_9_start)

print("End of Speed = 9")
with pd.option_context("display.max_columns", None):
    display(speed_9_end)

Static Speed Review
Trace             : trace_103
Target speed      : 9 km/h
Total target rows : 2,154
First occurrence  : 2017-11-23 13:15:06
Last occurrence   : 2017-11-23 13:55:06

Beginning of Speed = 9


,Row Description,Trace ID,Source Category,Source File,Timestamp,Longitude,Latitude,Speed,Operatorname,CellID,NetworkMode,RSRP,RSRQ,SNR,CQI,RSSI,DL_bitrate,UL_bitrate,State,NRxRSRP,NRxRSRQ,ServingCell_Lon,ServingCell_Lat,ServingCell_Distance
No.,,,,,,,,,,,,,,,,,,,,,,,,
1,Surrounding row,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:15:01,-8.500517,51.894671,0,A,2,LTE,-113.0,-13.0,7.0,6.0,-94.0,3724,69,D,-122.0,-17.0,-8.491719,51.893905,609.70
2,Surrounding row,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:15:02,-8.500517,51.894671,0,A,2,LTE,-108.0,-13.0,6.0,7.0,-91.0,4104,76,D,-121.0,-19.0,-8.491719,51.893905,609.70
3,Surrounding row,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:15:03,-8.500517,51.894671,0,A,2,LTE,-108.0,-13.0,6.0,8.0,-94.0,6028,112,D,-121.0,-19.0,-8.491719,51.893905,609.70
4,Surrounding row,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:15:04,-8.500517,51.894671,0,A,2,LTE,-111.0,-13.0,6.0,8.0,-94.0,1822,34,D,-121.0,-19.0,-8.491719,51.893905,609.70
5,Surrounding row,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:15:05,-8.500517,51.894671,0,A,2,LTE,-111.0,-13.0,6.0,8.0,-92.0,4205,98,D,-121.0,-19.0,-8.491719,51.893905,609.70
6,Surrounding row,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:15:06,-8.500517,51.894671,0,A,2,LTE,-110.0,-12.0,7.0,8.0,-92.0,3187,59,D,-122.0,-20.0,-8.491719,51.893905,609.70
7,Target: Speed = 9,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:15:06,-8.501135,51.894586,9,A,2,LTE,-110.0,-12.0,7.0,8.0,-92.0,3187,59,D,-122.0,-20.0,-8.491719,51.893905,650.55
8,Target: Speed = 9,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:15:07,-8.501135,51.894586,9,A,2,LTE,-110.0,-12.0,7.0,7.0,-94.0,2248,46,D,-122.0,-20.0,-8.491719,51.893905,650.55
9,Target: Speed = 9,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:15:08,-8.501135,51.894586,9,A,2,LTE,-113.0,-13.0,5.0,7.0,-94.0,2337,44,D,-121.0,-19.0,-8.491719,51.893905,650.55


End of Speed = 9


,Row Description,Trace ID,Source Category,Source File,Timestamp,Longitude,Latitude,Speed,Operatorname,CellID,NetworkMode,RSRP,RSRQ,SNR,CQI,RSSI,DL_bitrate,UL_bitrate,State,NRxRSRP,NRxRSRQ,ServingCell_Lon,ServingCell_Lat,ServingCell_Distance
No.,,,,,,,,,,,,,,,,,,,,,,,,
1,Target: Speed = 9,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:55:01,-8.501135,51.894586,9,A,2,LTE,-118.0,-13.0,3.0,6.0,-94.0,1509,28,D,-118.0,-18.0,-8.491719,51.893905,650.55
2,Target: Speed = 9,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:55:02,-8.501135,51.894586,9,A,2,LTE,-118.0,-13.0,3.0,6.0,-94.0,1006,20,D,-118.0,-18.0,-8.491719,51.893905,650.55
3,Target: Speed = 9,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:55:03,-8.501135,51.894586,9,A,2,LTE,-118.0,-13.0,3.0,6.0,-94.0,1431,30,D,-117.0,-17.0,-8.491719,51.893905,650.55
4,Target: Speed = 9,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:55:04,-8.501135,51.894586,9,A,2,LTE,-118.0,-13.0,3.0,6.0,-94.0,1062,20,D,-117.0,-17.0,-8.491719,51.893905,650.55
5,Target: Speed = 9,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:55:06,-8.501135,51.894586,9,A,2,LTE,-120.0,-14.0,2.0,6.0,-94.0,1599,50,D,-120.0,-18.0,-8.491719,51.893905,650.55
6,Surrounding row,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:55:06,-8.499688,51.894470,1,A,2,LTE,-120.0,-14.0,2.0,6.0,-94.0,1599,50,D,-120.0,-18.0,-8.491719,51.893905,550.43
7,Surrounding row,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:55:07,-8.499688,51.894470,1,A,2,LTE,-120.0,-14.0,2.0,6.0,-94.0,1431,27,D,-120.0,-18.0,-8.491719,51.893905,550.43
8,Surrounding row,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:55:08,-8.499688,51.894470,1,A,2,LTE,-120.0,-13.0,2.0,6.0,-94.0,626,21,D,-125.0,-19.0,-8.491719,51.893905,550.43
9,Surrounding row,trace_103,static,A_2017.11.23_13.14.40.csv,2017-11-23 13:55:09,-8.499688,51.894470,1,A,2,LTE,-120.0,-13.0,2.0,6.0,-94.0,1722,81,D,-125.0,-19.0,-8.491719,51.893905,550.43


In [47]:
static_change_check = (
    validated_df[
        validated_df["Source Category"].eq("static")
    ]
    .sort_values(
        ["Trace ID", "Timestamp"],
        kind="stable",
    )
    .copy()
)

static_change_check["Previous Speed"] = (
    static_change_check.groupby("Trace ID")["Speed"].shift()
)

static_change_check["Previous Longitude"] = (
    static_change_check.groupby("Trace ID")["Longitude"].shift()
)

static_change_check["Previous Latitude"] = (
    static_change_check.groupby("Trace ID")["Latitude"].shift()
)

static_change_check["Previous CellID"] = (
    static_change_check.groupby("Trace ID")["CellID"].shift()
)


comparable_rows = static_change_check[
    static_change_check["Previous Speed"].notna()
].copy()

speed_changed = (
    comparable_rows["Speed"]
    != comparable_rows["Previous Speed"]
)

coordinates_changed = (
    (comparable_rows["Longitude"]
     != comparable_rows["Previous Longitude"])
    | (comparable_rows["Latitude"]
       != comparable_rows["Previous Latitude"])
)

cell_id_changed = (
    comparable_rows["CellID"]
    != comparable_rows["Previous CellID"]
)


change_summary = pd.DataFrame(
    {
        "Observation": [
            "Speed and coordinates changed together",
            "Only Speed changed",
            "Only coordinates changed",
            "CellID changed",
        ],
        "Rows": [
            (speed_changed & coordinates_changed).sum(),
            (speed_changed & ~coordinates_changed).sum(),
            (~speed_changed & coordinates_changed).sum(),
            cell_id_changed.sum(),
        ],
    }
)

change_summary.index = range(1, len(change_summary) + 1)
change_summary.index.name = "No."

print("Static Movement Check")
print("Each row is compared with the previous row in the same trace.")
print(f"Rows compared: {len(comparable_rows):,}\n")

display(change_summary)

Static Movement Check
Each row is compared with the previous row in the same trace.
Rows compared: 15,234



,Observation,Rows
No.,,
1,Speed and coordinates changed together,20
2,Only Speed changed,0
3,Only coordinates changed,14
4,CellID changed,2


### Static Speed Finding

Among **15,249 static observations**, **5,218 rows** contain a non-zero
`Speed`. However, the row-by-row comparison shows only **20 actual speed
changes**, and every one occurred together with a coordinate change. There
were no cases where speed changed while both coordinates remained unchanged;
the coordinates also changed **14 times** without a speed change.

The detailed examples show a step-like pattern: a new combination of
`Longitude`, `Latitude`, and `Speed` appears, then repeats across many rows.
In `trace_102`, `CellID` remains `8` around the `2 → 97 → 27 km/h` sequence.
In `trace_103`, `CellID` remains `2` around the `0 → 9 → 1 km/h` sequence.

Across all static traces, `CellID` changed only **2 times**, so cell changes
were separate from most recorded GPS changes.

This pattern is more consistent with periodic GPS reporting than continuous
physical movement. The original `Speed` values remain unchanged, but they
should be interpreted cautiously in static traces.

---

## Findings So Far

- The **135 source traces** contain **174,523 raw observations** and share the
  same 20-column schema. Three source identifiers expand the working table to
  23 columns.
- Converting numeric `-` placeholders, removing **831 exact duplicate rows**,
  and parsing timestamps produces `cleaned_df` with **173,692 observations**.
- All **13,691 idle observations** have zero downlink bitrate. They remain in
  the prepared data, while the later throughput dataset will focus on
  `State = "D"`. The **4,087 downloading observations with zero throughput**
  remain meaningful active-download measurements.
- Context checks identified **229 invalid serving-cell RSRP measurements**,
  **22 unavailable neighbour-signal pairs**, and **1 additional invalid
  neighbour-quality measurement**. Only those measurement cells were changed
  to `NaN`; their rows were preserved.
- Non-zero speeds in static traces behave like repeated GPS readings rather
  than continuous physical movement, so the original values remain available
  with this limitation documented.

`combined_df` preserves the combined raw data, `cleaned_df` contains the basic
structural cleaning, and `validated_df` contains the confirmed domain-aware
measurement corrections.